# ECG Foundation Representation System
## Phase 2 Research Workspace: Temporal Representation Learning

This notebook demonstrates self-supervised pretraining strategies and supervised fine-tuning of the **Temporal Encoder**:
1. **Reconstruction Learning**: Compressing full ECG inputs and decoding them.
2. **Masked Autoencoder (MAE)**: Reconstructing randomly masked segments.
3. **Contrastive Learning (SimCLR)**: Maximizing cosine similarity between augmented views of the same record using a temperature-scaled InfoNCE loss.
4. **Supervised Fine-Tuning & Evaluation**: Multi-label classifier training, computing diagnostics metrics (Hamming Loss, Macro F1, Macro AUC).
5. **Explainability**: Generating gradient-based saliency maps over leads and time.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

print(f"Environment initialized. PyTorch version: {torch.__version__}")

## 1. Setup Toy Dataset
We generate synthetic 12-lead ECG signals ($N=128$, channels=12, length=1000) and multi-hot target labels ($5$ diagnostic categories) to simulate training loaders.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

# 128 samples, 12 leads, 1000 sequence steps
signals = np.random.randn(128, 12, 1000).astype(np.float32)
labels = np.random.randint(0, 2, (128, 5)).astype(np.float32)

# Create splits
train_ds = TensorDataset(torch.tensor(signals[:96]), torch.tensor(labels[:96]))
val_ds = TensorDataset(torch.tensor(signals[96:]), torch.tensor(labels[96:]))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

print(f"Train samples: {len(train_ds)} | Validation samples: {len(val_ds)}")

## 2. Initialize Model Components
We instantiate the `ECGBiLSTM` encoder and the companion `ECGReconstructionDecoder` (which maps the $256$-dimensional latent vector back to the original $12 	imes 1000$ waveform space).

In [ ]:
from temporal_encoder.encoder import ECGBiLSTM, ECGReconstructionDecoder

model = ECGBiLSTM(input_size=12, hidden_size=128, num_layers=2, num_classes=5)
decoder = ECGReconstructionDecoder(latent_dim=256, num_leads=12, signal_length=1000)

print("Encoder architecture:")
print(model)
print("\nDecoder architecture:")
print(decoder)

## 3. Pretraining Strategy 1: Reconstruction Learning
Encodes the full unmasked signal and trains the model to minimize the reconstruction Mean Squared Error (MSE).

In [ ]:
from temporal_encoder.strategies import ReconstructionLearningStrategy
from temporal_encoder.trainer import TemporalTrainer

trainer = TemporalTrainer(model, lr=1e-3)
recon_strategy = ReconstructionLearningStrategy()

print("Starting Reconstruction pretraining...")
recon_history = trainer.fit(
    train_loader=train_loader,
    epochs=5,
    is_pretraining=True,
    strategy=recon_strategy,
    decoder=decoder
)

plt.figure(figsize=(8, 4))
plt.plot(recon_history["train_loss"], marker='o', color="blue")
plt.title("Reconstruction Pretraining Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.show()

## 4. Pretraining Strategy 2: Masked Autoencoder (MAE)
Masks $30\%$ of the time-steps, processes visible signals, and decodes the masked regions, calculating MSE loss strictly over the masked points.

In [ ]:
from temporal_encoder.strategies import MaskedAutoencoderStrategy

mae_strategy = MaskedAutoencoderStrategy(mask_ratio=0.3)
mae_history = trainer.fit(
    train_loader=train_loader,
    epochs=5,
    is_pretraining=True,
    strategy=mae_strategy,
    decoder=decoder
)

plt.figure(figsize=(8, 4))
plt.plot(mae_history["train_loss"], marker='s', color="orange")
plt.title("Masked Autoencoder Pretraining Loss")
plt.xlabel("Epoch")
plt.ylabel("Masked MSE Loss")
plt.grid(True)
plt.show()

## 5. Pretraining Strategy 3: Contrastive Learning (SimCLR)
Applies Gaussian noise and scaling to create two views of each ECG signal, then projects their representation embeddings to a contrastive projection sphere and minimizes InfoNCE loss.

In [ ]:
from temporal_encoder.strategies import ContrastiveLearningStrategy

contrastive_strategy = ContrastiveLearningStrategy(temperature=0.1, projection_dim=64, latent_dim=256)
contrastive_history = trainer.fit(
    train_loader=train_loader,
    epochs=5,
    is_pretraining=True,
    strategy=contrastive_strategy
)

plt.figure(figsize=(8, 4))
plt.plot(contrastive_history["train_loss"], marker='^', color="green")
plt.title("Contrastive Pretraining Loss")
plt.xlabel("Epoch")
plt.ylabel("InfoNCE Loss")
plt.grid(True)
plt.show()

## 6. Supervised Downstream Fine-Tuning
We fine-tune the pretrained model end-to-end on multi-label targets using Binary Cross Entropy (BCE) with logits.

In [ ]:
supervised_history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    is_pretraining=False
)

plt.figure(figsize=(8, 4))
plt.plot(supervised_history["train_loss"], label="Train Loss", marker='o')
plt.plot(supervised_history["val_loss"], label="Val Loss", marker='x')
plt.title("Supervised Downstream Classification Fine-Tuning")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.legend()
plt.grid(True)
plt.show()

## 7. Model Evaluation
We run predictions on the validation loader and compute subset accuracy, Hamming Loss, Macro F1, and Macro ROC-AUC.

In [ ]:
from temporal_encoder.predictor import TemporalPredictor
from temporal_encoder.evaluator import TemporalEvaluator

predictor = TemporalPredictor(model)
val_probs = predictor.predict_proba(val_loader)

# Get ground truth labels
val_labels = []
for _, lbls in val_loader:
    val_labels.append(lbls.numpy())
val_labels = np.concatenate(val_labels, axis=0)

metrics = TemporalEvaluator.evaluate(val_labels, val_probs)
print("Validation Metrics:")
for k, v in metrics.items():
    print(f"- {k}: {v:.4f}")

## 8. Saliency Interpretability
We extract a gradient-based attribution map to highlight what leads and timestamps are most important for predicting the target diagnostic class.

In [ ]:
from temporal_encoder.explainer import TemporalSaliencyExplainer

explainer = TemporalSaliencyExplainer(model)
single_ecg = signals[0]

# Compute saliency attribution map for Class 0
saliency_map = explainer.explain(single_ecg, class_idx=0)

plt.figure(figsize=(15, 6))
plt.imshow(saliency_map, aspect='auto', cmap='hot', interpolation='nearest')
plt.colorbar(label='Gradient Magnitude Attribution')
plt.title("Lead-wise Temporal Saliency Map (Class 0)")
plt.xlabel("Time-steps")
plt.ylabel("Leads (0-11)")
plt.show()